# Federal procurement data preparation in Databricks

## Summary
The saved execution imported **5,804,212 procurement transaction rows and 297 columns**. The analytical projection retained **48 columns and all rows**. This is column selection, not a demonstrated improvement in accuracy or runtime.

**Contribution:** Julio Hernandez owned bronze-layer preparation and KPI definitions in the original team project. Ricardo owned silver/gold preparation and initial exploration; Felipe owned machine learning. Those teammates' implementations are outside this repository.

## Context and methods
The source is the FY2025 contract CSV snapshot referenced in the original project. One row is a transaction, not necessarily a distinct award. Code calculations and saved outputs below are preserved from the original notebook; headings and commentary were edited for readability. No Databricks rerun was performed for this edition.

### Assumptions and setup
- Use a Databricks Python notebook with Spark and a writable catalog/schema. The original runtime version was not recorded.
- Put the source CSV parts in a Unity Catalog volume, then update the `/Volumes/...` path and `workspace.usaspending` table names below to your own dedicated project namespace.
- Table writes use `overwrite`. Run only against your project tables.
- The date token `20251108` comes from the original filename. The exact download settings and execution timestamp were not retained.
- The source managed table retains all fields. The final projection removes identifiers; use the source table for deduplication or distinct-award calculations.

## Data preparation

### 1. Import CSV parts and save the source table

In [ ]:
df = (
    spark.read
        .option("header", True)          # CSV has header row
        .option("inferSchema", True)     # or define schema explicitly for safety
        .csv("/Volumes/workspace/usaspending/usa_cvs/FY2025_All_Contracts_Full_20251108_*.csv")
)

df.write.mode("overwrite").saveAsTable("workspace.usaspending.FY2025_all_contracts")

### 2. Inspect source dimensions

In [ ]:
from pyspark.sql import functions as F, types as T
import pandas as pd
import matplotlib.pyplot as plt

table = "workspace.usaspending.fy2025_all_contracts"
df = spark.table(table)
n_rows = df.count()
n_cols = len(df.columns)
print((n_rows, n_cols))

(5804212, 297)


### 3. Profile null counts by column

This checks SQL NULL values; blank strings and coded missing values need separate rules.

In [ ]:
missing_rows = []
for c in df.columns:
    missing_rows.append(
        f"SELECT '{c}' AS column_name, "
        f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS missing "
        f"FROM {table}"
    )
missing_query = " UNION ALL ".join(missing_rows)
missing_df = spark.sql(missing_query)
display(missing_df)

column_name,missing
contract_transaction_unique_key,0
contract_award_unique_key,0
award_id_piid,0
modification_number,0
transaction_number,297947
parent_award_agency_id,1185246
parent_award_agency_name,1185246
parent_award_id_piid,1185246
parent_award_modification_number,1185246
federal_action_obligation,0


### 4. Measure dominant-value frequency

The dominant-value fraction uses non-null values as its denominator. Window functions rank each column's value frequencies.

In [ ]:
mode_rows = []
for c in df.columns:
    mode_rows.append(f"""
    SELECT
      '{c}' AS column_name,
      mode_percent
    FROM (
      SELECT
        value,
        cnt,
        cnt / SUM(cnt) OVER () AS mode_percent,
        ROW_NUMBER() OVER (ORDER BY cnt DESC) AS rn
      FROM (
        SELECT `{c}` AS value, COUNT(*) AS cnt
        FROM {table}
        WHERE `{c}` IS NOT NULL
        GROUP BY `{c}`
      ) base
    ) ranked
    WHERE rn = 1
    """)
mode_query = " UNION ALL ".join(mode_rows)
mode_df = spark.sql(mode_query)
display(mode_df)

column_name,mode_percent
contract_transaction_unique_key,1.722886758788273E-7
contract_award_unique_key,7.597930606256284E-5
award_id_piid,8.097567766304883E-5
modification_number,0.8002326586279068
transaction_number,0.9948958141317209
parent_award_agency_id,0.6645957125469206
parent_award_agency_name,0.6645957125469206
parent_award_id_piid,0.03705288153236027
parent_award_modification_number,0.7249457995577365
federal_action_obligation,0.11009487592803295


### 5. Select columns for the analytical projection

Original rules: drop columns with more than 4,000,000 nulls, dominant non-null value share above 95%, identifier/code name patterns, or explicit exclusions. The absolute null threshold is snapshot-specific; it should be revisited for a new dataset. Low variance alone does not prove that a field has no business value.

In [ ]:
m_missing = 4000000
n_mode_percent = 0.95

cols_missing = [
    r["column_name"]
    for r in missing_df.filter(F.col("missing") > m_missing).collect()
]
cols_mode = [
    r["column_name"]
    for r in mode_df.filter(F.col("mode_percent") > n_mode_percent).collect()
]
cols_pattern = [
    c for c in df.columns
    if "_id" in c.lower() or "_code" in c.lower()
]
cols_manual = [
    "contract_transaction_unique_key",
    "contract_award_unique_key",
    "modification_number",
    "parent_award_modification_number",
    "treasury_accounts_funding_this_award",
    "federal_accounts_funding_this_award",
    "object_classes_funding_this_award",
    "program_activities_funding_this_award",
    "recipient_uei",
    "recipient_name_raw",
    "recipient_parent_uei",
    "recipient_parent_name_raw",
    "prime_award_transaction_recipient_cd_original",
    "prime_award_transaction_recipient_cd_current",
    "recipient_phone_number",
    "recipient_fax_number",
    "prime_award_transaction_place_of_performance_cd_original",
    "prime_award_transaction_place_of_performance_cd_current",
    "alaskan_native_corporation_owned_firm",
    "commercial_item_acquisition_procedures",
    "clinger_cohen_act_planning",
    "construction_wage_rate_requirements",
    "corporate_entity_not_tax_exempt",
    "corporate_entity_tax_exempt",
    "epa_designated_product",
    "fed_biz_opps",
    "labor_standards",
    "limited_liability_corporation",
    "materials_supplies_articles_equipment",
    "minority_owned_business",
    "native_american_owned_business",
    "performance_based_service_acquisition",
    "recovered_materials_sustainability",
    "self_certified_small_disadvantaged_business",
    "subchapter_scorporation",
    "subcontracting_plan",
    "the_ability_one_program",
    "type_of_contract_pricing",
    "usaspending_permalink",
    "veteran_owned_business",
    "woman_owned_business",
    "women_owned_small_business"
]

cols_to_drop = list(set(cols_missing + cols_mode + cols_pattern + cols_manual))

print(f"Dropping {len(cols_to_drop)} columns:")

Dropping 249 columns:


### 6. Check projected dimensions

In [ ]:
df_clean = df.drop(*cols_to_drop)
n_rows = df_clean.count()
n_cols = len(df_clean.columns)
print((n_rows, n_cols))

(5804212, 48)


### 7. Save the analytical table

In [ ]:
clean_table = "workspace.usaspending.FY2025_all_contracts_bronze"
df_clean.write.mode("overwrite").saveAsTable(clean_table)

## Takeaways
This notebook demonstrates multi-file ingestion, managed-table writes, data profiling, SQL aggregation, and window functions. Row counts match before and after projection, but duplicate transactions, coded missing values, and source coverage were not independently verified. Full diagnostic tables are also exported in `data/`.

Continue with [KPI analysis](02_kpi_analysis.ipynb). See [limitations](../docs/limitations.md) for grain and data-quality considerations.